In [1]:
import torch
import torch.nn as nn
import torch.nn.functional as F
import timm

In [2]:
model_name = "vit_base_patch16_224"
pretrained = True
dropout = 0.1
num_classes = 2

In [8]:
class ViTFaceAntiSpoofing(nn.Module):
    def __init__(self):
        super().__init__()
        self.vit = timm.create_model(model_name, pretrained=pretrained, num_classes=0)
        embed_dim = self.vit.num_features if hasattr(self.vit, 'num_features') else 768
        
        self.classifier = nn.Sequential(
            nn.LayerNorm(embed_dim),
            nn.Dropout(dropout),
            nn.Linear(embed_dim, 512),
            nn.GELU(),
            nn.Dropout(dropout),
            nn.Linear(512, num_classes)
        )
    
    def forward(self, x):
        features = self.vit(x)
        return self.classifier(features)

In [9]:
model = ViTFaceAntiSpoofing()

In [10]:
sum(p.numel() for p in model.parameters())

86194946

In [11]:
x = torch.randn(5, 3, 224, 224)
outputs = model(x)

In [12]:
outputs.shape

torch.Size([5, 2])